In [1]:
import ccxt
import pandas as pd

exchange = ccxt.binance()
# Fetch last 1000 candles of 5-minute data
ohlcv = exchange.fetch_ohlcv('DOGE/USDT', timeframe='5m', limit=10000)

df = pd.DataFrame(ohlcv, columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Vol'])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

In [2]:
df

,timestamp,Open,High,Low,Close,Vol
0,2025-12-25 19:10:00,0.12736,0.12738,0.12732,0.12734,348685.0
1,2025-12-25 19:15:00,0.12733,0.12734,0.12714,0.12715,456476.0
2,2025-12-25 19:20:00,0.12714,0.12723,0.12700,0.12701,1456150.0
3,2025-12-25 19:25:00,0.12702,0.12709,0.12701,0.12704,742776.0
4,2025-12-25 19:30:00,0.12704,0.12705,0.12694,0.12705,534801.0
...,...,...,...,...,...,...
995,2025-12-29 06:05:00,0.12667,0.12702,0.12667,0.12701,1740150.0
996,2025-12-29 06:10:00,0.12701,0.12705,0.12700,0.12701,354946.0
997,2025-12-29 06:15:00,0.12701,0.12722,0.12701,0.12716,1145983.0
998,2025-12-29 06:20:00,0.12715,0.12718,0.12698,0.12702,1228990.0


In [3]:
import ccxt
import pandas as pd
import time

exchange = ccxt.binance()
symbol = 'DOGE/USDT'
timeframe = '5m'
limit = 1000  # Max allowed per request
target_total = 100000
all_ohlcv = []

# Calculate start time (5 mins * 10,000 candles ago)
# Time in milliseconds: target_total * 5 * 60 * 1000
since = exchange.milliseconds() - (target_total * 5 * 60 * 1000)

print(f"Fetching {target_total} candles...")

while len(all_ohlcv) < target_total:
    try:
        # Fetch a chunk
        new_ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since, limit)
        if not new_ohlcv:
            break
            
        all_ohlcv.extend(new_ohlcv)
        
        # Update 'since' to the timestamp of the last candle + 1ms to avoid overlap
        since = new_ohlcv[-1][0] + 1
        
        print(f"Downloaded {len(all_ohlcv)}/{target_total}...")
        
        # Respect rate limits
        time.sleep(exchange.rateLimit / 1000)
        
    except Exception as e:
        print(f"Error: {e}")
        break

# Convert to DataFrame
df = pd.DataFrame(all_ohlcv, columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Vol'])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

# Trim to exactly target_total if needed
df = df.iloc[-target_total:]

print("\nFinal Data:")
print(df.head())

Fetching 100000 candles...
Downloaded 1000/100000...
Downloaded 2000/100000...
Downloaded 3000/100000...
Downloaded 4000/100000...
Downloaded 5000/100000...
Downloaded 6000/100000...
Downloaded 7000/100000...
Downloaded 8000/100000...
Downloaded 9000/100000...
Downloaded 10000/100000...
Downloaded 11000/100000...
Downloaded 12000/100000...
Downloaded 13000/100000...
Downloaded 14000/100000...
Downloaded 15000/100000...
Downloaded 16000/100000...
Downloaded 17000/100000...
Downloaded 18000/100000...
Downloaded 19000/100000...
Downloaded 20000/100000...
Downloaded 21000/100000...
Downloaded 22000/100000...
Downloaded 23000/100000...
Downloaded 24000/100000...
Downloaded 25000/100000...
Downloaded 26000/100000...
Downloaded 27000/100000...
Downloaded 28000/100000...
Downloaded 29000/100000...
Downloaded 30000/100000...
Downloaded 31000/100000...
Downloaded 32000/100000...
Downloaded 33000/100000...
Downloaded 34000/100000...
Downloaded 35000/100000...
Downloaded 36000/100000...
Downloaded

In [4]:
df.to_csv("DOGE_raw.csv")